# Language-Aware Retrieval

**Multilingual Semantic Search and BM25-Based Hybrid Retrieval**

This notebook implements Origin's selected language-aware retrieval pipeline for identifying historical graduation projects related to newly proposed project ideas.

The system uses different retrieval strategies based on the query language:

- **Arabic Queries:** Chunked Semantic Search using multilingual Sentence Transformer embeddings and Cosine Similarity.
- **English Queries:** Hybrid Search combining BM25 keyword-based retrieval and Chunked Semantic Search through Reciprocal Rank Fusion (RRF).

Token-based chunking is applied to preserve information from longer project descriptions and queries, while RRF combines lexical and semantic rankings for English retrieval.

The retrieval configuration was selected based on development-set experiments documented in `06_Hybrid_Search_Experiments.ipynb`.

This notebook includes the selected retrieval implementation, validation, development evaluation, and artifact export. Independent evaluation on the reserved final test set will be conducted in `07_Model_Evaluation.ipynb`.


## 1. Setup and Dependencies

Install the libraries required for BM25, semantic embeddings, and hybrid retrieval.

In [1]:
%pip install -q sentence-transformers rank-bm25

In [2]:
import pandas as pd
import numpy as np
import re
import torch

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

print("Libraries imported successfully!")
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

Libraries imported successfully!
Device: cuda


## 2. Load and Validate Dataset

Load the cleaned graduation projects dataset from GitHub and verify its structure before hybrid retrieval.

In [3]:
# Load dataset
url = (
    "https://raw.githubusercontent.com/"
    "esrrahafase-crypto/Origin/main/"
    "data/processed/projects_clean.csv"
)

df = pd.read_csv(url, encoding="utf-8-sig")

# Standardize columns
df.columns = df.columns.str.strip().str.lower()
df["department"] = df["department"].str.strip()

# Validate dataset
assert df.shape == (103, 7)
assert df["project_id"].is_unique
assert df[["title", "abstract"]].notna().all().all()
assert df[["title", "abstract"]].apply(
    lambda col: col.astype(str).str.strip().ne("").all()
).all()

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("Departments:", df["department"].nunique())

display(df.head(3))

Dataset loaded successfully!
Shape: (103, 7)
Departments: 6


,project_id,title,abstract,department,year,keywords,source_page
0,P001,Automatic Plant Watering and Tracking System,Many people struggle with watering their plant...,Computer Science,2024 – 2025,"Moisturem Air humidity, AWTS system, Waterin, ...",5
1,P002,علامة CourseAssist: CLO Performance Assistant ...,The rapid evolution of educational technology ...,Computer Science,2024 – 2025,"Program Learning Outcomes, Course Learning Out...",6
2,P003,واصف Mobile Application for Deaf and Mute indi...,"Our System, Wasif, offers a communication appl...",Computer Science,2024 – 2025,"Arabic Sign Language, Sign Language Gestures i...",7



## 3. Text Preparation and BM25 Preprocessing

This section combines project titles and abstracts and applies consistent lexical preprocessing for BM25.

The preprocessing includes Unicode normalization, lowercasing, tokenization, and English stop-word filtering.

The original project text remains unchanged for semantic retrieval. Arabic-to-English translation is performed later at query time for the BM25 component.


In [4]:

# ============================================================
# 3. TEXT PREPARATION AND BM25 PREPROCESSING
# ============================================================

import re
import unicodedata
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

hybrid_df = df.copy()

# Original text for semantic retrieval
hybrid_df["text"] = (
    hybrid_df["title"].astype(str).str.strip()
    + ". "
    + hybrid_df["abstract"].astype(str).str.strip()
)

hybrid_df["text"] = (
    hybrid_df["text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

project_texts = hybrid_df["text"].tolist()

# Consistent BM25 preprocessing
def tokenize(text):

    text = unicodedata.normalize(
        "NFKC", str(text)
    ).casefold()

    tokens = re.findall(r"(?u)\b\w+\b", text)

    tokens = [
        token for token in tokens
        if len(token) >= 2
        and token not in ENGLISH_STOP_WORDS
    ]

    return tokens


# Build tokenized historical corpus
tokenized_corpus = [
    tokenize(text)
    for text in project_texts
]

assert len(tokenized_corpus) == 103
assert all(len(tokens) > 0 for tokens in tokenized_corpus)

print("BM25 PREPROCESSING COMPLETED")
print("=" * 55)
print("Total projects:", len(project_texts))
print("Tokenized documents:", len(tokenized_corpus))
print("Sample tokens:", tokenized_corpus[0][:20])


BM25 PREPROCESSING COMPLETED
Total projects: 103
Tokenized documents: 103
Sample tokens: ['automatic', 'plant', 'watering', 'tracking', 'people', 'struggle', 'watering', 'plants', 'regularly', 'usually', 'leads', 'plants', 'unhealthy', 'dying', 'address', 'common', 'issue', 'important', 'monitor', 'basic']


## 4. BM25 Retrieval Model

Build a BM25 index using the tokenized graduation project descriptions.

In [5]:
# Build BM25 index
bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index created successfully!")
print("Indexed projects:", len(tokenized_corpus))

BM25 index created successfully!
Indexed projects: 103


## 5. BM25 Retrieval Function

Implement keyword-based retrieval using BM25 scores to rank the most relevant graduation projects.

In [6]:
def bm25_search(title, description, top_k=5):
    # Validate input
    if not title.strip() or not description.strip():
        raise ValueError("Title and description must not be empty.")

    if not 1 <= top_k <= len(hybrid_df):
        raise ValueError("Invalid top_k value.")

    # Prepare English query
    query_text = f"{title.strip()}. {description.strip()}"
    query_tokens = tokenize(query_text)

    # Calculate BM25 scores
    scores = bm25.get_scores(query_tokens)

    # Keep projects with positive lexical scores
    matched_indices = np.flatnonzero(scores > 0)

    result_columns = [
        "rank", "project_id", "title", "abstract",
        "department", "source_page", "bm25_score"
    ]

    if len(matched_indices) == 0:
        return pd.DataFrame(columns=result_columns)

    sorted_indices = matched_indices[
        np.argsort(-scores[matched_indices], kind="stable")
    ][:top_k]

    results = hybrid_df.iloc[sorted_indices][
        ["project_id", "title", "abstract", "department", "source_page"]
    ].copy()

    results.insert(0, "rank", range(1, len(results) + 1))
    results["bm25_score"] = scores[sorted_indices]

    return results.reset_index(drop=True)


print("English BM25 retrieval is ready!")


English BM25 retrieval is ready!


## 6. BM25 Retrieval Testing

Test BM25 retrieval using the same English project idea evaluated in the Semantic Search notebook.

In [7]:
# Test English query
test_title = "AI-Based Early Detection of Parkinson's Disease"

test_description = (
    "A mobile application that uses machine learning "
    "and wearable sensor data to detect early signs "
    "of Parkinson's disease and monitor patient symptoms."
)

bm25_results = bm25_search(
    title=test_title,
    description=test_description,
    top_k=5
)

display(
    bm25_results[
        ["rank", "project_id", "title", "bm25_score"]
    ].round(4)
)

,rank,project_id,title,bm25_score
0,1,P025,Early Detection and Monitoring of Parkinson’s ...,48.7279
1,2,P005,Thabat – ثبات,30.5957
2,3,P080,A Smart Mobile Application For Skin Cancer Det...,17.6335
3,4,P026,Potato Guard,15.9575
4,5,P083,Remote Delivery of Healthcare Services for Man...,13.2784


## 7. Load Semantic Retrieval Model

Load the same pretrained multilingual Sentence Transformer used in the standalone Semantic Search implementation.

In [8]:
# Load semantic model
model_name = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(model_name, device=device)

print("Semantic model loaded successfully!")
print("Device:", device)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Semantic model loaded successfully!
Device: cuda
Embedding dimension: 768


/tmp/ipykernel_41142/3673033430.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


## 8. Token-Based Project Chunking

Split graduation project descriptions into overlapping token-safe chunks to preserve information beyond the model's 128-token limit.

In [9]:
# Configure chunking
max_tokens = model.max_seq_length
special_tokens = model.tokenizer.num_special_tokens_to_add(pair=False)

chunk_size = max_tokens - special_tokens
chunk_overlap = 20
chunk_step = chunk_size - chunk_overlap

assert chunk_step > 0

# Split text into token-safe chunks
def safe_chunk_text(text, overlap=20):
    token_ids = model.tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False
    )

    chunks = []
    start = 0

    while start < len(token_ids):
        end = min(
            start + model.max_seq_length - special_tokens,
            len(token_ids)
        )

        while end > start:
            chunk = model.tokenizer.decode(
                token_ids[start:end],
                skip_special_tokens=True
            )

            length = len(model.tokenizer.encode(
                chunk,
                add_special_tokens=True,
                truncation=False
            ))

            if length <= model.max_seq_length:
                break

            end -= 1

        if end == start:
            raise ValueError("Unable to create a valid chunk.")

        chunks.append(chunk)

        if end == len(token_ids):
            break

        start = max(start + 1, end - overlap)

    return chunks


# Chunk all project descriptions
project_chunk_texts = [
    safe_chunk_text(text)
    for text in project_texts
]

all_chunk_texts = [
    chunk
    for chunks in project_chunk_texts
    for chunk in chunks
]

# Validate chunk lengths
assert all(
    len(model.tokenizer.encode(
        chunk,
        add_special_tokens=True,
        truncation=False
    )) <= max_tokens
    for chunk in all_chunk_texts
)

print("Project chunking completed successfully!")
print("Total projects:", len(project_chunk_texts))
print("Total chunks:", len(all_chunk_texts))
print("Average chunks per project:",
      round(np.mean([len(c) for c in project_chunk_texts]), 2))
print("Maximum tokens:", max_tokens)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (208 > 128). Running this sequence through the model will result in indexing errors


Project chunking completed successfully!
Total projects: 103
Total chunks: 298
Average chunks per project: 2.89
Maximum tokens: 128


## 8.1. Generate and Aggregate Chunk Embeddings

In [10]:
# Generate embeddings for all project chunks
chunk_embeddings = model.encode(
    all_chunk_texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Aggregate chunk embeddings per project
chunked_project_embeddings = []

start = 0

for chunks in project_chunk_texts:
    count = len(chunks)

    project_vectors = chunk_embeddings[start:start + count]
    project_embedding = project_vectors.mean(axis=0)

    project_embedding /= np.linalg.norm(project_embedding)

    chunked_project_embeddings.append(project_embedding)
    start += count

chunked_project_embeddings = np.vstack(chunked_project_embeddings)

# Validate embeddings
assert start == len(chunk_embeddings)
assert chunked_project_embeddings.shape == (103, 768)
assert np.isfinite(chunked_project_embeddings).all()

print("Chunked project embeddings generated successfully!")
print("Embeddings shape:", chunked_project_embeddings.shape)

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Chunked project embeddings generated successfully!
Embeddings shape: (103, 768)


## 9. Semantic Retrieval Function

Implement semantic retrieval using normalized Sentence Transformer embeddings and Cosine Similarity.

In [11]:

def semantic_search(title, description, top_k=5):
    # Validate input
    if not title.strip() or not description.strip():
        raise ValueError("Title and description must not be empty.")

    if not 1 <= top_k <= len(hybrid_df):
        raise ValueError("Invalid top_k value.")

    # Prepare and chunk query
    query_text = f"{title.strip()}. {description.strip()}"
    query_chunks = safe_chunk_text(query_text)

    # Generate query chunk embeddings
    query_vectors = model.encode(
        query_chunks,
        batch_size=16,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Aggregate query embeddings
    query_embedding = query_vectors.mean(axis=0, keepdims=True)
    query_embedding /= np.linalg.norm(
        query_embedding, axis=1, keepdims=True
    )

    # Calculate cosine similarity
    scores = cosine_similarity(
        query_embedding,
        chunked_project_embeddings
    )[0]

    # Retrieve Top-K
    top_indices = np.argsort(-scores)[:top_k]

    results = hybrid_df.iloc[top_indices][
        ["project_id", "title", "abstract", "department", "source_page"]
    ].copy()

    results.insert(0, "rank", range(1, len(results) + 1))
    results["similarity_score"] = scores[top_indices]

    return results.reset_index(drop=True)


print("Chunked semantic retrieval is ready!")


Chunked semantic retrieval is ready!


## 10. Reciprocal Rank Fusion (RRF)

Combine BM25 and semantic retrieval rankings using Reciprocal Rank Fusion to produce a unified hybrid ranking.

In [12]:

# ============================================================
# 10. HYBRID RETRIEVAL WITH RRF
# ============================================================

def hybrid_search(title, description, top_k=5, rrf_k=60):

    if not title.strip() or not description.strip():
        raise ValueError(
            "Title and description must not be empty."
        )

    if not 1 <= top_k <= len(hybrid_df):
        raise ValueError("Invalid top_k.")

    if rrf_k <= 0:
        raise ValueError("rrf_k must be positive.")

    # 1. Semantic retrieval uses original language
    semantic_results = semantic_search(
        title,
        description,
        top_k=len(hybrid_df)
    )

    semantic_ranks = dict(zip(
        semantic_results["project_id"],
        semantic_results["rank"]
    ))

    # 2. BM25 performs Arabic translation internally
    bm25_results = bm25_search(
        title,
        description,
        top_k=len(hybrid_df)
    )

    bm25_ranks = dict(zip(
        bm25_results["project_id"],
        bm25_results["rank"]
    ))

    # 3. Prepare final project-level ranking
    results = hybrid_df[
        [
            "project_id",
            "title",
            "abstract",
            "department",
            "source_page"
        ]
    ].copy()

    results["semantic_rank"] = (
        results["project_id"].map(semantic_ranks)
    )

    results["bm25_rank"] = (
        results["project_id"].map(bm25_ranks)
    )

    # 4. Reciprocal Rank Fusion
    semantic_rrf = (
        1.0 / (rrf_k + results["semantic_rank"])
    )

    bm25_rrf = (
        1.0 / (rrf_k + results["bm25_rank"])
    ).fillna(0.0)

    results["rrf_score"] = semantic_rrf + bm25_rrf

    results["retrieval_mode"] = (
        "hybrid"
        if len(bm25_results) > 0
        else "semantic_fallback"
    )

    # 5. Rank projects
    results = results.sort_values(
        ["rrf_score", "project_id"],
        ascending=[False, True]
    ).head(top_k).copy()

    results.insert(
        0, "rank", range(1, len(results) + 1)
    )

    return results.reset_index(drop=True)


print("Hybrid search updated successfully!")


Hybrid search updated successfully!



## 11. Language-Aware Retrieval

The final development-stage retrieval strategy selects the search method based on the query language:

- **Arabic:** Chunked Semantic Search.
- **English:** Hybrid Search combining BM25 and Chunked Semantic Search using RRF.

This configuration was selected based on development-set evaluation results.


In [13]:
def language_aware_search(title, description, language, top_k=5):
    # Select retrieval strategy by language
    if language == "ar":
        results = semantic_search(
            title, description, top_k=top_k
        )
        results["retrieval_mode"] = "semantic_ar"

    elif language == "en":
        results = hybrid_search(
            title, description, top_k=top_k
        )
        results["retrieval_mode"] = "hybrid_en"

    else:
        raise ValueError("Unsupported language.")

    return results


print("Language-aware retrieval is ready!")


Language-aware retrieval is ready!



## 12. English Language-Aware Retrieval Testing

Test the selected English retrieval strategy using Hybrid Search with BM25, Chunked Semantic Search, and RRF.


In [14]:
# Test English language-aware retrieval
english_results = language_aware_search(
    title=test_title,
    description=test_description,
    language="en",
    top_k=5
)

# Validate retrieval strategy
assert len(english_results) == 5
assert english_results["project_id"].is_unique
assert english_results["retrieval_mode"].eq("hybrid_en").all()
assert english_results["rrf_score"].is_monotonic_decreasing

print("English language-aware retrieval passed!")
print("Retrieval mode:", english_results["retrieval_mode"].iloc[0])

display(
    english_results[
        [
            "rank",
            "project_id",
            "title",
            "bm25_rank",
            "semantic_rank",
            "rrf_score"
        ]
    ].round(4)
)


English language-aware retrieval passed!
Retrieval mode: hybrid_en


,rank,project_id,title,bm25_rank,semantic_rank,rrf_score
0,1,P025,Early Detection and Monitoring of Parkinson’s ...,1.0,1,0.0328
1,2,P005,Thabat – ثبات,2.0,3,0.0320
2,3,P080,A Smart Mobile Application For Skin Cancer Det...,3.0,2,0.0320
3,4,P032,دُرَّة DURRA – Ovarian Cancer Diagnosis Using ...,7.0,4,0.0306
4,5,P082,Predicting Diabetes Patients Based On Machine ...,6.0,5,0.0305



## 13. Arabic Language-Aware Retrieval Testing

Test the selected Arabic retrieval strategy using Chunked Semantic Search without BM25 contribution.


In [15]:
# Test Arabic language-aware retrieval
arabic_title = "الكشف المبكر عن مرض باركنسون باستخدام الذكاء الاصطناعي"

arabic_description = (
    "تطبيق يستخدم تعلم الآلة وبيانات أجهزة الاستشعار "
    "القابلة للارتداء للكشف المبكر عن مرض باركنسون "
    "ومراقبة أعراض المرض لدى المرضى."
)

arabic_results = language_aware_search(
    title=arabic_title,
    description=arabic_description,
    language="ar",
    top_k=5
)

# Validate retrieval strategy
assert len(arabic_results) == 5
assert arabic_results["project_id"].is_unique
assert arabic_results["retrieval_mode"].eq("semantic_ar").all()
assert arabic_results["similarity_score"].is_monotonic_decreasing

print("Arabic language-aware retrieval passed!")
print("Retrieval mode:", arabic_results["retrieval_mode"].iloc[0])

display(
    arabic_results[
        [
            "rank",
            "project_id",
            "title",
            "similarity_score"
        ]
    ].round(4)
)


Arabic language-aware retrieval passed!
Retrieval mode: semantic_ar


,rank,project_id,title,similarity_score
0,1,P082,Predicting Diabetes Patients Based On Machine ...,0.6584
1,2,P025,Early Detection and Monitoring of Parkinson’s ...,0.6342
2,3,P032,دُرَّة DURRA – Ovarian Cancer Diagnosis Using ...,0.5788
3,4,P080,A Smart Mobile Application For Skin Cancer Det...,0.5604
4,5,P093,Deep Learning Based Wound Classification And T...,0.5512



## 14. Language-Aware Retrieval Validation

Validate the selected retrieval strategy for Arabic and English queries, including result rankings, uniqueness, finite scores, and language-specific retrieval modes.


In [16]:
# Validate retrieval results
for results in [english_results, arabic_results]:
    assert len(results) == 5
    assert results["project_id"].is_unique
    assert results["rank"].tolist() == [1, 2, 3, 4, 5]

# Validate English hybrid retrieval
assert english_results["retrieval_mode"].eq("hybrid_en").all()
assert english_results["rrf_score"].is_monotonic_decreasing
assert np.isfinite(english_results["rrf_score"]).all()

# Validate Arabic semantic retrieval
assert arabic_results["retrieval_mode"].eq("semantic_ar").all()
assert arabic_results["similarity_score"].is_monotonic_decreasing
assert np.isfinite(arabic_results["similarity_score"]).all()

# Validate embedding representations
assert chunked_project_embeddings.shape == (
    len(hybrid_df),
    model.get_sentence_embedding_dimension()
)
assert np.isfinite(chunked_project_embeddings).all()

print("Language-aware retrieval validation passed!")
print("English mode:", english_results["retrieval_mode"].iloc[0])
print("Arabic mode:", arabic_results["retrieval_mode"].iloc[0])
print("All retrieval results and embeddings are valid.")


Language-aware retrieval validation passed!
English mode: hybrid_en
Arabic mode: semantic_ar
All retrieval results and embeddings are valid.


/tmp/ipykernel_41142/4110497602.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()



## 15. Save Language-Aware Retrieval Outputs

Save the selected retrieval configuration, chunked project embeddings, project ID mapping, and Arabic and English retrieval results for reproducibility and future RAG integration.


In [17]:
from pathlib import Path
import json
import numpy as np

# Create output directory
output_dir = Path("/content/origin_retrieval_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Save language-aware retrieval results
english_results.to_csv(
    output_dir / "english_retrieval_results.csv",
    index=False,
    encoding="utf-8-sig"
)

arabic_results.to_csv(
    output_dir / "arabic_retrieval_results.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save chunked project embeddings
np.save(
    output_dir / "chunked_project_embeddings.npy",
    chunked_project_embeddings
)

# Save project ID mapping
hybrid_df[["project_id"]].to_csv(
    output_dir / "embedding_project_ids.csv",
    index=False
)

# Save selected retrieval configuration
retrieval_config = {
    "semantic_model": model_name,
    "max_sequence_length": model.max_seq_length,
    "chunk_overlap": chunk_overlap,
    "chunk_aggregation": "mean_pooling_l2_normalized",
    "arabic_strategy": "chunked_semantic",
    "english_strategy": "bm25_semantic_rrf",
    "rrf_k": 60,
    "semantic_weight": 1.0,
    "bm25_weight": 1.0,
    "selection_split": "dev",
    "top_k": 5
}

with open(
    output_dir / "retrieval_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        retrieval_config,
        f,
        indent=4,
        ensure_ascii=False
    )

# Validate saved artifacts
expected_files = {
    "english_retrieval_results.csv",
    "arabic_retrieval_results.csv",
    "chunked_project_embeddings.npy",
    "embedding_project_ids.csv",
    "retrieval_config.json"
}

assert {file.name for file in output_dir.iterdir()} == expected_files
assert chunked_project_embeddings.shape[0] == len(hybrid_df)

print("Language-aware retrieval outputs saved successfully!")

for file in sorted(output_dir.iterdir()):
    print(file.name)


Language-aware retrieval outputs saved successfully!
arabic_retrieval_results.csv
chunked_project_embeddings.npy
embedding_project_ids.csv
english_retrieval_results.csv
retrieval_config.json


In [18]:
import shutil
from google.colab import files

# Create ZIP archive
archive_path = shutil.make_archive(
    "/content/origin_retrieval_outputs",
    "zip",
    root_dir=output_dir
)

# Download retrieval artifacts
files.download(archive_path)

print("Language-aware retrieval artifacts downloaded successfully!")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Language-aware retrieval artifacts downloaded successfully!



## 16. Language-Aware Development Evaluation

Evaluate the selected language-aware retrieval strategy on the 40 development queries using human-selected reference project IDs.

Reference Hit@5 and MRR@5 are calculated only for queries with available reference annotations.


In [19]:
# Load development queries
eval_url = (
    "https://raw.githubusercontent.com/"
    "esrrahafase-crypto/Origin/main/"
    "data/evaluation/evaluation_queries.csv"
)

eval_df = pd.read_csv(eval_url, encoding="utf-8-sig")

dev_queries = eval_df[
    eval_df["split"] == "dev"
].copy().reset_index(drop=True)

assert len(dev_queries) == 40

# Evaluate the selected retrieval strategy
evaluation_records = []

for _, query in dev_queries.iterrows():
    results = language_aware_search(
        title=query["query_title"],
        description=query["query_description"],
        language=query["language"],
        top_k=5
    )

    refs = query["reference_project_ids"]

    if pd.isna(refs) or not str(refs).strip():
        continue

    reference_ids = {
        ref.strip()
        for ref in str(refs).split(";")
        if ref.strip()
    }

    matched_ranks = results.loc[
        results["project_id"].isin(reference_ids),
        "rank"
    ]

    first_rank = (
        int(matched_ranks.min())
        if not matched_ranks.empty else None
    )

    evaluation_records.append({
        "query_id": query["query_id"],
        "language": query["language"],
        "reference_hit_at_5": int(first_rank is not None),
        "reference_mrr_at_5": (
            1.0 / first_rank if first_rank is not None else 0.0
        )
    })

language_aware_dev_eval = pd.DataFrame(evaluation_records)

# Summarize reference-based performance
dev_evaluation_summary = (
    language_aware_dev_eval
    .groupby("language")
    .agg(
        evaluated_queries=("query_id", "count"),
        reference_hit_at_5=("reference_hit_at_5", "mean"),
        reference_mrr_at_5=("reference_mrr_at_5", "mean")
    )
    .reset_index()
)

# Validate evaluation coverage
assert len(language_aware_dev_eval) == 26
assert dev_evaluation_summary["evaluated_queries"].eq(13).all()

print("Language-aware development evaluation completed!")
print("Development queries:", len(dev_queries))
print("Queries with references:", len(language_aware_dev_eval))

display(dev_evaluation_summary.round(4))


Language-aware development evaluation completed!
Development queries: 40
Queries with references: 26


,language,evaluated_queries,reference_hit_at_5,reference_mrr_at_5
0,ar,13,0.9231,0.6141
1,en,13,0.9231,0.8462



## 17. Conclusions

This notebook implemented Origin's selected language-aware retrieval pipeline for identifying historical graduation projects related to newly proposed project ideas.

### Final Retrieval Architecture

The system combines multilingual semantic retrieval, BM25, and Reciprocal Rank Fusion (RRF), with the retrieval strategy selected according to the query language.

- **Arabic queries:** Chunked Semantic Search using multilingual Sentence Transformer embeddings and Cosine Similarity.
- **English queries:** Hybrid Search combining BM25 and Chunked Semantic Search using RRF (`k=60`, equal fusion weights).

The semantic component uses token-safe overlapping chunks, mean pooling, and L2 normalization to represent complete project descriptions and incoming queries.


Arabic-to-English query translation was evaluated during development but excluded from the selected retrieval pipeline, as standalone Chunked Semantic Search achieved better Arabic retrieval performance on the development set.

### Development Evaluation

The selected approach was evaluated using 40 development queries: 20 Arabic and 20 English. Reference-based metrics were calculated for 26 queries with human-selected reference project IDs.

| Metric | Arabic | English |
|---|---:|---:|
| Evaluated Queries | 13 | 13 |
| Reference Hit@5 | 0.9231 | 0.9231 |
| Reference MRR@5 | 0.6141 | 0.8462 |

The language-aware configuration preserved the strongest observed development performance for each language among the evaluated retrieval strategies.

The experiments supporting this selection, including Standard RRF, Weighted RRF, and Arabic error analysis, are documented separately in `06_Hybrid_Search_Experiments.ipynb`.

### Saved Artifacts

The notebook exports:

- Arabic and English sample retrieval results.
- Chunked project embeddings.
- Project ID mapping.
- Retrieval configuration in JSON format.

These artifacts support reproducibility and future integration with Origin's RAG pipeline.

### Limitations and Next Steps

- The retrieval strategy was selected using development data, so its performance on unseen queries has not yet been established.
- Human-selected reference IDs may not cover every relevant historical project.
- The selected configuration must remain fixed during independent final-test evaluation.
- The retrieval pipeline will be integrated with an LLM through Retrieval-Augmented Generation (RAG) to produce evidence-grounded suggestions for improving graduation project ideas.
- Generated suggestions will require separate evaluation for relevance, faithfulness, and practical usefulness.

The language-aware retrieval pipeline is Origin's selected development-stage implementation, pending evaluation on the reserved final test set.
